In [62]:
# Setup and package imports 
import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder,
    VectorAssembler, MinMaxScaler
)

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import GBTClassifier

from xgboost.spark import SparkXGBClassifier


StatementMeta(, b70643e2-d63b-48a2-a78a-6978b199f6bf, 88, Finished, Available, Finished, False)

In [63]:
# -------------------------------------------
# a. Read Dataset and create Spark DataFrame
# -------------------------------------------

spark_df = spark.read.table('FraudDetection.dbo.silver_layer_ciferfrauddata_2') #.select(features)
display( spark_df.limit(5) )

StatementMeta(, b70643e2-d63b-48a2-a78a-6978b199f6bf, 89, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c740f5a-7fc4-4997-b75d-b863459dccd1)

In [65]:
# ** 4. Train / Test Split **
train_df, test_df = spark_df.randomSplit([0.8, 0.2], seed=42)

train_df.cache()
test_df.cache()

StatementMeta(, b70643e2-d63b-48a2-a78a-6978b199f6bf, 91, Finished, Available, Finished, False)

DataFrame[step: bigint, type: string, amount: double, oldbalanceOrg: double, newbalanceOrig: double, oldbalanceDest: double, newbalanceDest: double, isFraud: bigint, orig_balance_diff: double, dest_balance_diff: double, orig_zero: int, dest_zero: int, log_amount: double, amount_to_balance: double, orig_error: int, dest_error: int, orig_name_type: string, dest_name_type: string, orig_name_freq: bigint, dest_name_freq: bigint, pair_freq: bigint]

In [66]:
%run /ml_common

StatementMeta(, b70643e2-d63b-48a2-a78a-6978b199f6bf, 96, Finished, Available, Finished, True)

In [78]:
# -----------------------------------------
# 0. Get categorical and numerical features
# --------------------------------------------
label_col = "isFraud" 
categorical_cols, numeric_cols = auto_detect_colum_type(spark_df, label_col)

# **1. Preprocessing the pipeline** 
# ---------------------------------------
# 1.1 Encode categorical freatures 
# --------------------------------------
indexers, encoders = encode_categorical_features(categorical_cols)

# -------------------------------------------------------
# 1.2 Assemble features 
# ----------------------------
assembler_inputs = [f"{c}_ohe" for c in categorical_cols] + numeric_cols
print('Assembler Inputs: ', assembler_inputs)

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw"
)

# ----------------------------
# 1.3 Scaling: MimMax scaling 
# ----------------------------
scaler = MinMaxScaler(
    inputCol="features_raw",
    outputCol="features"
)


xgb = SparkXGBClassifier(
    features_col="features",
    label_col=label_col,
    prediction_col="prediction",
    probability_col="probability",
    num_workers=4
)

# **3. Define MLflow pipeline **
pipeline = Pipeline(
    stages=[
        *indexers,
        *encoders,
        assembler,
        scaler,
        xgb
    ]
)

# ** 4. Hyperparamer tunning: for  **
paramGrid = hyperparameter_tunning(xgb, 'xgboost')

# ** 5. Evaluate and validate the model **
evaluator, cv = evaluate_and_validate(label_col, pipeline, paramGrid)

StatementMeta(, b70643e2-d63b-48a2-a78a-6978b199f6bf, 108, Finished, Available, Finished, False)

Categorical: ['type', 'orig_name_type', 'dest_name_type']
Numeric: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_diff', 'dest_balance_diff', 'orig_zero', 'dest_zero', 'log_amount', 'amount_to_balance', 'orig_error', 'dest_error', 'orig_name_freq', 'dest_name_freq', 'pair_freq']
Assembler Inputs:  ['type_ohe', 'orig_name_type_ohe', 'dest_name_type_ohe', 'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_diff', 'dest_balance_diff', 'orig_zero', 'dest_zero', 'log_amount', 'amount_to_balance', 'orig_error', 'dest_error', 'orig_name_freq', 'dest_name_freq', 'pair_freq']


In [79]:
# -----------------------------------
# 4. MLflow tracking (Fabric auto-integrated)
# -----------------------------------
mlflow.set_experiment("fabric_spark_xgboostClassifier")

with mlflow.start_run():

    model = cv.fit(train_df)

    best_model = model.bestModel

    # Log best params manually
    best_lr = best_model.stages[-1]

    #mlflow.log_param("regParam", best_lr._java_obj.getRegParam())
    #mlflow.log_param("elasticNetParam", best_lr._java_obj.getElasticNetParam())
    #mlflow.log_param("maxIter", best_lr._java_obj.getMaxIter())

    # Evaluate
    preds = best_model.transform(test_df)
    auc = evaluator.evaluate(preds)

    mlflow.log_metric("AUC", auc)

    print("Best AUC:", auc)

StatementMeta(, b70643e2-d63b-48a2-a78a-6978b199f6bf, 109, Finished, Available, Finished, False)

2026-05-30 22:38:05,711 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 4 workers with
	booster params: {'device': 'cpu', 'max_depth': 7, 'objective': 'binary:logistic', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-30 22:38:05,712 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 4 workers with
	booster params: {'device': 'cpu', 'max_depth': 5, 'objective': 'binary:logistic', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-30 22:38:05,738 INFO XGBoost-PySpark: _fit Running xgboost-2.0.3 on 4 workers with
	booster params: {'device': 'cpu', 'max_depth': 5, 'objective': 'binary:logistic', 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-05-30 22:38:05,750 INFO XGBoost-PySpark: _fit Running xgboost-2.0.

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.scheduler.BarrierJobRunWithDynamicAllocationException: [SPARK-24942]: Barrier execution mode does not support dynamic resource allocation for now. You can disable dynamic resource allocation by setting Spark conf "spark.dynamicAllocation.enabled" to "false".
	at org.apache.spark.errors.SparkCoreErrors$.barrierStageWithDynamicAllocationError(SparkCoreErrors.scala:229)
	at org.apache.spark.scheduler.DAGScheduler.checkBarrierStageWithDynamicAllocation(DAGScheduler.scala:598)
	at org.apache.spark.scheduler.DAGScheduler.createResultStage(DAGScheduler.scala:692)
	at org.apache.spark.scheduler.DAGScheduler.handleJobSubmitted(DAGScheduler.scala:1408)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3289)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3278)
	at org.apache.spark.scheduler.DAG